# 13 - Institutional Scalping R&D Playbook for MasterVP and KenKem

This notebook is the learning spine for the revised build plan. It is written for the exact problem in this repo:

- **MasterVP**: tick-volume-profile breakout first, failed-breakout/local-value reversion second.
- **KenKem**: EMA/ADX/RSI entry ensemble with better entry selectivity, structural stops, and dynamic targets.
- **Symbols**: XAUUSD first, BTCUSD only after symbol-specific cost/session/parity work.
- **Production target**: not a nice backtest; a release process where every claim survives parity, realistic costs, overfitting controls, portfolio risk, and forward monitoring.

The purpose is to teach the thinking process, vocabulary, and gates so you can read the build plan without missing the quant logic behind each task.

## 0. How to study this path

Read the notebooks in this order:

1. `00_raw_tick_data.ipynb` through `04_labeling.ipynb`: data contract, bars, features, labels.
2. `05_first_look_eda.ipynb` through `08_discovery.ipynb`: what the data says before a strategy exists.
3. `09_quick_backtest.ipynb` through `12_overfitting_and_drawdown_honesty.ipynb`: costs, parity, drawdown, overfitting.
4. This notebook: how to turn the tools into an institutional-grade R&D and release program.

The mental sequence is always:

**data quality -> causal features -> path-aware labels -> edge autopsy -> cheap backtest -> tick engine -> MT5 parity -> cost stress -> WF/CPCV/MC -> DSR/PBO -> portfolio risk -> forward-test.**

## 1. The core question

A retail backtest usually asks: **which settings made the most money?**

A serious quant R&D process asks harder questions:

- What is the economic mechanism?
- Is the edge visible before realized P&L, for example in MFE/MAE/path behavior?
- Did we choose the idea because it won one sample, or because it survived many independent tests?
- Does the edge survive realistic costs, latency, spread shocks, and broker execution?
- Does the live EA reproduce the research engine?
- Does the strategy improve the portfolio after correlation, tail overlap, and shared drawdown risk?
- What signal tells us to stop trading it live?

If a strategy cannot answer these questions, it is not production-grade even if the backtest PF looks good.

## 2. Strategy glossary - the words you need to master

| Term | Meaning | Why it matters here |
|---|---|---|
| Auction / value area | Market rotates around prices where activity concentrates. | MasterVP is a volume-profile auction strategy. |
| POC / VAH / VAL | Point of control, value-area high, value-area low. | Breakout and reversion levels. |
| Rolling VP | Volume profile over a trailing lookback. | Current MasterVP basis; useful but can be structurally naive. |
| Anchored VP | Profile started at a session open, swing, news event, or regime boundary. | More institutional way to define the auction being broken. |
| First breakout | First valid expansion from value into imbalance. | MasterVP's primary book. |
| Failed breakout | Breakout that cannot continue and returns into value. | Candidate for MasterVP reversion book. |
| Order-flow imbalance | Net pressure from buy/sell initiated flow or tick-direction proxy. | We lack L2 order book, but tick-direction proxies can still diagnose flow. |
| Flow toxicity | Adverse selection: informed/aggressive flow makes passive fills dangerous. | VPIN/order-flow research informs why some breakouts fail. |
| MFE / MAE | Maximum favorable/adverse excursion after entry. | Model-free way to study edge before exit rules distort P&L. |
| R multiple | Profit/loss measured in units of initial risk. | Normalizes XAU vs BTC and different stop sizes. |
| Meta-label | Secondary model that decides whether to accept/size a primary signal. | Useful for MasterVP breakout quality, not for inventing direction. |
| Regime | Market state: trend, chop, compression, expansion, high-cost, etc. | Most failed sweeps are hidden regime problems. |
| MinTRL | Minimum track record length needed for statistical confidence. | KenKem is sample-constrained; filters can break the gate by cutting too many trades. |
| DSR / PSR / PBO | Deflated Sharpe, probabilistic Sharpe, probability of backtest overfitting. | Protects against false locks from many trials. |
| CPCV | Combinatorial purged cross-validation. | Tests many OOS paths while controlling leakage. |
| Purge / embargo | Remove train samples overlapping or near test samples. | Prevents serial-correlation leakage in financial data. |
| HRP | Hierarchical risk parity. | Better than naive mean-variance when samples are small and streams are correlated. |
| Drift monitor | Live expected-vs-realized scorecard. | Tells when an edge is decaying or execution changed. |

## 3. MasterVP - what the strategy really is

MasterVP should be treated as two separate books:

### Book A: Breakout from value

Hypothesis: when price leaves a meaningful VP value area with enough trend/flow support, the next auction seeks a new value area before reverting.

Research variables:

- Distance from entry to VAH/VAL/POC in ATR.
- Distance to prior session VP levels.
- Node density above/below entry.
- Runway to next high-volume node.
- Profile shape: balanced, skewed, double-distribution, thin tail.
- Time-of-day and spread regime.
- Trend persistence and ATR percentile.
- MFE path after entry before any exit rule.

### Book B: Failed breakout / local-value reversion

Hypothesis: a breakout that fails to attract continuation and returns into value can be faded back to local POC/value.

This is not the same as generic countertrend trading. It needs a failure state:

1. Valid breakout beyond VAH/VAL.
2. No continuation within N bars or insufficient MFE.
3. Return into value or loss of flow support.
4. Target local POC/value mean, not a random fixed RR.

In [ ]:
# MasterVP breakout event schema - use this as the target export contract.
mastervp_event_columns = [
    "strategy", "symbol", "tf", "entry_ts", "side",
    "entry_price", "initial_sl", "initial_risk",
    "rolling_vah", "rolling_val", "rolling_poc",
    "session_vah", "session_val", "session_poc",
    "dist_vah_atr", "dist_val_atr", "dist_poc_atr",
    "runway_to_next_node_atr", "node_density_above", "node_density_below",
    "profile_shape", "atr_pct", "adx", "di_spread",
    "spread_atr", "session_id", "hour_utc",
    "mfe_r", "mae_r", "reach_1r", "exit_tag", "realized_r",
]
mastervp_event_columns

## 4. KenKem - what needs to become less naive

KenKem currently has a real XAU M1 edge, but its weak points are structural:

- Entries are rule families switched on/off, not conditioned by regime.
- Stop distance can be a generic ATR/fixed-distance construct instead of true invalidation.
- RR targets are often fixed or loosely dynamic, not tied to expected MFE or nearby structure.
- BTC cannot be trusted until pip/point thresholds become ATR/spread-relative.

The key is not to add more indicators. The key is to ask for each entry family:

- What market state is E1/E2/E4/E5 supposed to exploit?
- What invalidates the idea?
- Where should the trade reasonably take profit?
- Which regime makes it stop working?
- Does it add independent sample or just noisy extra trades?

In [ ]:
# KenKem entry-family audit schema.
kenkem_event_columns = [
    "strategy", "symbol", "tf", "entry_type", "entry_ts", "side",
    "ema_stack_state", "ema_compression", "cross_age", "touch_age",
    "adx", "di_spread", "trend_quality", "sideways_score", "atr_pct",
    "distance_to_ema200_atr", "distance_to_vp_level_atr",
    "session_id", "hour_utc", "spread_atr",
    "initial_sl_type", "initial_sl_atr", "target_type", "target_r",
    "mfe_r", "mae_r", "reach_1r", "exit_tag", "realized_r",
]
kenkem_event_columns

## 5. The institutional research gate

Every hypothesis should pass this checklist before it is allowed to become a lock:

| Gate | Question | Fail condition |
|---|---|---|
| Data gate | Are ticks/bars/features valid and causal? | Data gaps, bad spread, lookahead, bar mismatch. |
| Mechanism gate | Is there a believable market reason? | Pure parameter luck or no economic story. |
| Edge-autopsy gate | Is edge visible in MFE/MAE/path metrics? | Only realized P&L improves because exit model flatters it. |
| Parity gate | Does C++/MQL/MT5 reproduce the same behavior? | Config mismatch, entry mismatch, P&L mismatch. |
| Cost gate | Does it survive realistic spread, commission, slippage, latency? | PF dies under plausible costs. |
| Robustness gate | WF/CPCV/MC/DSR/PBO/MinTRL pass. | Single fold, single peak, too few trades. |
| Portfolio gate | Does it improve the book after correlation and DD overlap? | It duplicates risk or increases tail drawdown. |
| Production gate | Can live monitoring detect drift and stop it? | No telemetry or kill rule. |

A strategy that passes only the backtest gate is not a strategy yet. It is a research lead.

## 6. Cost and execution thinking

For scalping, cost realism is not a detail. It is the strategy.

Costs to model:

- Spread paid at entry/exit.
- Commission per lot and per side.
- Slippage, including adverse slippage on high-volatility bars.
- Latency: fill at tick N+k, not the decision tick.
- Rollover/swap if positions cross broker rollover.
- Weekend BTC spread/gap behavior.
- Requote/rejection approximation if spread exceeds allowable threshold.

Useful tests:

- Breakeven cost per trade: how much extra cost kills expectancy?
- Cost stress curve: PF as cost increases.
- Session-specific cost stress: Asia, London, NY, rollover, weekend.
- Regime-specific cost stress: high ATR, news-like spread spikes, low liquidity.

In [ ]:
# Cost stress pseudocode. Wire this to normalized trade streams later.
import numpy as np


def stress_pnl(pnl_usd, extra_cost_per_trade):
    pnl_usd = np.asarray(pnl_usd, dtype=float)
    return pnl_usd - extra_cost_per_trade


def profit_factor(pnl):
    pnl = np.asarray(pnl, dtype=float)
    gross_win = pnl[pnl > 0].sum()
    gross_loss = -pnl[pnl < 0].sum()
    return np.inf if gross_loss == 0 else gross_win / gross_loss

# Example placeholder:
# for c in np.linspace(0, 50, 26):
#     stressed = stress_pnl(trades["pnl_usd"], c)
#     print(c, stressed.sum(), profit_factor(stressed))

## 7. Regime thinking

Most strategy settings fail because the strategy is averaging good and bad regimes.

Useful regime axes:

- Volatility: ATR percentile, realized volatility, spread/ATR.
- Trend persistence: ADX, DI spread, slope consistency, Hurst proxy.
- Auction state: balanced VP, skewed VP, thin tail, multi-distribution profile.
- Session: Asia, London, NY, rollover, weekend BTC.
- Path state: after a win/loss cluster, current drawdown, current day P&L.
- Liquidity state: tick count, spread, gap frequency.

The deployable output should be a simple matrix:

| Stream | Regime | Trade? | Size multiplier | Reason |
|---|---|---:|---:|---|
| MasterVP breakout | trend + expansion + acceptable spread | yes | 1.0 | Breakouts continue. |
| MasterVP breakout | balanced chop + low runway | no | 0.0 | Breakouts fail. |
| MasterVP reversion | failed breakout back into value | yes | 0.5 | Low-correlation book. |
| KenKem E2 | chop / weak trend | no | 0.0 | Stalls and EA exits leak. |

## 8. Portfolio thinking

You do not release an EA. You release a book of risk.

Portfolio questions:

- What is the daily correlation between MasterVP and KenKem?
- What is the correlation during the worst 5% of days?
- Do drawdowns overlap?
- Which stream contributes most component risk?
- What happens if XAU trend chops for three months?
- What happens if BTC weekend spreads triple?
- What is the account-level daily DD if two charts lose at once?

Default allocation method:

- Start equal risk.
- Compare inverse variance and HRP.
- Cap fractional Kelly hard; do not allow Kelly to dominate on thin samples.
- Run allocation choice through CPCV/PBO like a strategy parameter.

## 9. Release scorecard template

Before release, fill this table.

| Section | Required answer |
|---|---|
| Strategy | MasterVP breakout / MasterVP reversion / KenKem E1/E2/E4/E5 / portfolio. |
| Symbol/TF | XAUUSD M5, XAUUSD M1, BTCUSD M1/M3/M5, etc. |
| Mechanism | Why should this make money? |
| Data | Tick coverage, gaps, spread realism, bar parity. |
| Costs | Spread, commission, slippage, latency, swap assumptions. |
| Trials | Number of configs tested and `sr_trial_std`. |
| Validation | WF, CPCV, MC, PSR, DSR, PBO, MinTRL. |
| MT5 confirm | Exact run path and diff result. |
| Portfolio impact | Weight, component risk, tail correlation, DD overlap. |
| Live controls | Lot cap, daily DD, total open risk, spread guard, kill switch. |
| Drift monitor | What metric triggers review or shutdown? |
| Decision | Release / demo only / reject / needs more data. |

## 10. Common traps in this repo

- **Config mismatch before code bug.** Diff `.set` vs MT5 echo first.
- **Indented `.set` files.** MT5 can ignore indented settings; keep flush-left `key=value`.
- **C++ exit optimism.** Entry-side sweeps can be useful; exit-side locks need MT5.
- **BTC pip scaling.** XAU pip thresholds do not port to BTC. Convert decision thresholds to ATR/spread units first.
- **Node-net value parity gap.** VAH/VAL/POC distances are safer; node-net/absorption requires parity proof.
- **Small KenKem sample.** Cutting trades can make PF look better but fail MinTRL.
- **Pooled OOS trap.** A pooled win can hide a dead recent fold.
- **Top-10-winner dependence.** If top winners explain all net, test tail dependence and regime concentration.
- **Portfolio double-counting.** Two EAs on the same XAU account can breach book-level DD even if each obeys its own cap.

## 11. Reading list mapped to this project

- **Deflated Sharpe Ratio** - use when choosing from many tested configs.  
  Bailey & Lopez de Prado: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551

- **Probability of Backtest Overfitting / CSCV** - use when you have many trials and want PBO.  
  Bailey, Borwein, Lopez de Prado & Zhu: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2326253

- **Hierarchical Risk Parity** - use for portfolio allocation with short/noisy samples.  
  Lopez de Prado: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678

- **Flow toxicity / VPIN** - useful lens for why some flow regimes are dangerous.  
  Easley, Lopez de Prado & O'Hara: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1695596

- **Order-flow imbalance and price impact** - maps order-flow imbalance to short-horizon price moves.  
  Cont, Kukanov & Stoikov: https://arxiv.org/abs/1011.6402

- **High-frequency trading market design** - helps separate what MT5 CFD scalping can and cannot emulate.  
  Budish, Cramton & Shim: https://academic.oup.com/qje/article/130/4/1547/1916146

## 12. What to do next

Open `docs/BUILD-PLAN.md`. Start at Phase 1, not with another parameter sweep.

The most important upgrades are:

1. Experiment registry.
2. Unified trade schema.
3. Realistic cost/latency model.
4. Exit-model calibration.
5. MasterVP anchored/session VP event taxonomy.
6. KenKem structural stop and dynamic target audit.
7. Regime matrix.
8. Portfolio/account-level risk governor.

That is the path from a profitable EA to a research and production program.